In [ ]:
#Step 1: Preparing your dataset directory structure

In [1]:
import os

# Alphabet folders
alphabet_folders = [f'dataset/{c}' for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ']

# 10 selected word folders
word_folders = [
    'dataset/HELLO',
    'dataset/THANKYOU',
    'dataset/YES',
    'dataset/NO',
    'dataset/HELP',
    'dataset/PLEASE',
    'dataset/WATER',
    'dataset/FOOD',
    'dataset/GOOD',
    'dataset/BAD'
]

# Combine all
folders = ['dataset'] + alphabet_folders + word_folders

# Create folders
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Alphabets + 10 word folders created!")

✅ Alphabets + 10 word folders created!


In [2]:
import os 
print(os.listdir())

['.ipynb_checkpoints', 'dataset', 'Untitled.ipynb']


In [4]:
!pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
#Step 2: Collecting your dataset images.

In [ ]:
import cv2
import os
import time
import uuid
import string

# -----------------------------
# Classes (Alphabets + Words)
# -----------------------------
alphabets = list(string.ascii_uppercase)

words = [
    'HELLO','THANKYOU','YES','NO','HELP',
    'PLEASE','WATER','FOOD','GOOD','BAD'
]

labels = alphabets + words

# Number of images per class
number_imgs = 20

# Base dataset path
IMAGES_PATH = 'dataset'

# ROI box
x1, y1, x2, y2 = 100, 100, 300, 300

for label in labels:
    
    folder_path = os.path.join(IMAGES_PATH, label)
    os.makedirs(folder_path, exist_ok=True)

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("❌ Camera not accessible")
        break

    print(f"\n👉 Collecting images for: {label}")
    print("⏳ Get ready...")
    time.sleep(3)

    for imgnum in range(number_imgs):
        ret, frame = cap.read()
        
        if not ret:
            print("❌ Failed to grab frame")
            break

        frame = cv2.flip(frame, 1)

        # ROI crop
        roi = frame[y1:y2, x1:x2]

        # Save image
        imgname = os.path.join(folder_path, f"{label}.{uuid.uuid1()}.jpg")
        cv2.imwrite(imgname, roi)

        # Show frame
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, f'{label} ({imgnum+1}/{number_imgs})',
                    (10,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        cv2.imshow('Frame', frame)
        cv2.imshow('ROI', roi)

        # Wait before next capture
        time.sleep(1)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

print("✅ Dataset collection completed!")


👉 Collecting images for: A
⏳ Get ready...
❌ Failed to grab frame

👉 Collecting images for: B
⏳ Get ready...
❌ Failed to grab frame

👉 Collecting images for: C
⏳ Get ready...
❌ Failed to grab frame

👉 Collecting images for: D
⏳ Get ready...

👉 Collecting images for: E
⏳ Get ready...

👉 Collecting images for: F
⏳ Get ready...

👉 Collecting images for: G
⏳ Get ready...

👉 Collecting images for: H
⏳ Get ready...

👉 Collecting images for: I
⏳ Get ready...

👉 Collecting images for: J
⏳ Get ready...

👉 Collecting images for: K
⏳ Get ready...

👉 Collecting images for: L
⏳ Get ready...

👉 Collecting images for: M
⏳ Get ready...


In [ ]:
import tensorflow as tf

IMG_SIZE = (64, 64)
BATCH_SIZE = 32

dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = dataset.class_names
print("Classes:", class_names)

Found 740 files belonging to 36 classes.
Classes: ['A', 'B', 'BAD', 'C', 'D', 'E', 'F', 'FOOD', 'G', 'GOOD', 'H', 'HELLO', 'HELP', 'I', 'J', 'K', 'L', 'M', 'N', 'NO', 'O', 'P', 'PLEASE', 'Q', 'R', 'S', 'T', 'THANKYOU', 'U', 'V', 'W', 'WATER', 'X', 'Y', 'YES', 'Z']


In [5]:
train_size = int(0.8 * len(dataset))

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size)

In [6]:
train_ds = train_ds.map(lambda x, y: (x/255.0, y))
val_ds = val_ds.map(lambda x, y: (x/255.0, y))

In [7]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))

In [9]:
from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.0444 - loss: 3.5874 - val_accuracy: 0.0682 - val_loss: 3.5810
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - accuracy: 0.0543 - loss: 3.5762 - val_accuracy: 0.0606 - val_loss: 3.5453
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.0592 - loss: 3.5084 - val_accuracy: 0.0909 - val_loss: 3.4409
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.0855 - loss: 3.3854 - val_accuracy: 0.0985 - val_loss: 3.1991
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.1217 - loss: 3.1396 - val_accuracy: 0.1136 - val_loss: 2.7971
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - accuracy: 0.1184 - loss: 3.0367 - val_accuracy: 0.2500 - val_loss: 2.6214
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.2039 - loss: 2.7571 - val_accuracy: 0.2045 - val_loss: 2.7557
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.2171 - loss: 2.6506 - val_accuracy: 0.1970 - 

In [1]:
from tensorflow.keras.models import load_model
import numpy as np
import cv2

model = load_model("best_sign_model.h5")

In [2]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\ASUS\Documents\sign_language_hindi
['.ipynb_checkpoints', 'best_sign_model.h5', 'class_names.json', 'dataset', 'Hindi.ttf', 'hindi_text.png', 'sign_language_hindi.py.ipynb', 'Untitled.ipynb']


In [3]:
model.save("best_sign_model.h5")

In [5]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_sign_model.h5', 'class_names.json', 'dataset', 'Hindi.ttf', 'hindi_text.png', 'sign_language_hindi.py.ipynb', 'Untitled.ipynb']


In [4]:
from tensorflow.keras.models import load_model

model = load_model("best_sign_model.h5")
print("Model loaded successfully!")

Model loaded successfully!


In [6]:
#Live Prediction

In [7]:
from PIL import ImageFont

font_path = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf.ttf"
font = ImageFont.truetype(font_path, 32)
print("Font loaded successfully!")

OSError: cannot open resource

In [ ]:
pred = model.predict(img)
pred_class = class_names[np.argmax(pred)]
confidence = np.max(pred)
print(pred_class, confidence)

In [16]:
from PIL import Image, ImageDraw, ImageFont

# Create a blank image
img = Image.new('RGB', (500, 200), color='white')

# Prepare to draw
draw = ImageDraw.Draw(img)

# Use your loaded font
text = "नमस्ते दुनिया"  # Hindi for "Hello World"
draw.text((50, 50), text, font=font, fill='black')

# Save the image
img.save("hindi_text.png")
print("Image saved with Hindi text!")

Image saved with Hindi text!


In [8]:
font_path = r"C:\Windows\Fonts\Mangal.ttf"

In [9]:
import os
print(os.path.exists(font_path))  # Must print True

False


In [10]:
import os

folder = r"C:\Users\ASUS\Documents\sign_language_hindi"
for file in os.listdir(folder):
    if file.lower().endswith(".ttf"):
        print(file)

Hindi.ttf


In [11]:
import os

folder = r"C:\Users\ASUS\Documents\sign_language_hindi"
files = os.listdir(folder)
print(files)

['.ipynb_checkpoints', 'best_sign_model.h5', 'class_names.json', 'dataset', 'Hindi.ttf', 'hindi_text.png', 'sign_language_hindi.py.ipynb', 'Untitled.ipynb']


In [12]:
from PIL import ImageFont

font_path = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"

# Load the font
font = ImageFont.truetype(font_path, 32)
print("Font loaded successfully!")

Font loaded successfully!


In [13]:
from PIL import ImageFont

font_path = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"

# Verify Python sees it
import os
print(os.path.exists(font_path))  # Should print True

True


In [14]:
FONT_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"
font = ImageFont.truetype(FONT_PATH, 32)

In [15]:
import os
font_path = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"
print(os.path.exists(font_path))  # should print True

True


In [17]:
# Font loading
FONT_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"
font = ImageFont.truetype(FONT_PATH, 32)

# Preprocess frame for RGB model
IMG_SIZE = (64, 64)
img = cv2.resize(frame, IMG_SIZE)
img = img / 255.0
img = np.expand_dims(img, axis=0)  # shape (1,64,64,3)

NameError: name 'frame' is not defined

In [18]:
MODEL_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\best_sign_model.h5"
model = load_model(MODEL_PATH)

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       1,179,904 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 37)                  │           9,509 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,282,663 (4.89 MB)

 Trainable params: 1,282,661 (4.89 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from PIL import ImageFont, ImageDraw, Image
import os

# ===============================
# Sign → Hindi Mapping
# ===============================
SIGN_TO_HINDI = {
    'A': 'अ', 'B': 'ब', 'BAD': 'बुरा', 'C': 'च', 'D': 'ड', 'E': 'ए', 'F': 'फ',
    'FOOD': 'खाना', 'G': 'ग', 'GOOD': 'अच्छा', 'H': 'ह', 'HELLO': 'नमस्ते',
    'HELP': 'मदद', 'I': 'इ', 'J': 'ज', 'K': 'क', 'L': 'ल', 'M': 'म', 'N': 'न',
    'NO': 'नहीं', 'O': 'ओ', 'P': 'प', 'PLEASE': 'कृपया', 'Q': 'क्ष', 'R': 'र',
    'S': 'स', 'T': 'त', 'THANKYOU': 'धन्यवाद', 'U': 'उ', 'V': 'व', 'W': 'व्ह',
    'WATER': 'पानी', 'X': 'श', 'Y': 'य', 'YES': 'हाँ', 'Z': 'ज्ञ'
}

# ===============================
# Load trained model
# ===============================
MODEL_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\best_sign_model.h5"
model = load_model(MODEL_PATH)
print("Model loaded successfully!")

# Make sure class order matches training
class_names = sorted(SIGN_TO_HINDI.keys())

# ===============================
# Load Hindi font with fallback
# ===============================
def load_hindi_font(path, size=32):
    if os.path.exists(path):
        try:
            font = ImageFont.truetype(path, size)
            print(f"Loaded font: {path}")
            return font
        except OSError:
            print(f"Failed to load font: {path}. Using default font.")
    else:
        print(f"Font file not found at {path}. Using default font.")
    return ImageFont.load_default()

FONT_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf"
font = load_hindi_font(FONT_PATH, 32)

# ===============================
# Start webcam for live prediction
# ===============================
cap = cv2.VideoCapture(0)
IMG_SIZE = (64, 64)  # model input size

# Prediction smoothing variables
prev_prediction = ""
smooth_counter = 0
SMOOTH_FRAMES = 5  # number of consistent frames required

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Preprocess frame for RGB model
    img = cv2.resize(frame, IMG_SIZE)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)  # shape (1,64,64,3)

    # Predict
    predictions = model.predict(img, verbose=0)
    class_idx = np.argmax(predictions[0])
    sign = class_names[class_idx]

    # Smooth predictions over multiple frames
    if sign == prev_prediction:
        smooth_counter += 1
    else:
        smooth_counter = 0
        prev_prediction = sign

    if smooth_counter >= SMOOTH_FRAMES:
        hindi_text = SIGN_TO_HINDI.get(sign, "")
        display_text = f"{sign} → {hindi_text}"
    else:
        display_text = ""

    # Convert frame to PIL Image to draw Hindi text
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)
    draw = ImageDraw.Draw(pil_img)
    draw.text((10, 50), display_text, font=font, fill=(0, 255, 0))

    # Convert back to OpenCV image
    frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    # Display
    cv2.imshow("Sign Language Hindi Translator", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()

Model loaded successfully!
Loaded font: C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf


In [ ]:
from PIL import Image, ImageDraw

# Create a blank image
img = Image.new('RGB', (500, 200), color='white')
draw = ImageDraw.Draw(img)

# Draw Hindi text
text = "नमस्ते दुनिया"
draw.text((50, 50), text, font=font, fill='black')

# Save the image
img.save(r"C:\Users\ASUS\Documents\sign_language_hindi\hindi_text.png")
print("Image saved with Hindi text!")

In [ ]:
# Preprocess frame for RGB model
IMG_SIZE = (64, 64)
img = cv2.resize(frame, IMG_SIZE)  # resize to model input
img = img / 255.0                  # normalize
img = np.expand_dims(img, axis=0)  # shape (1, 64, 64, 3) for RGB

In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Dataset path
DATASET_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\dataset"

IMG_SIZE = (64, 64)
BATCH_SIZE = 32

# ImageDataGenerator for augmentation + normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # 20% for validation
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

num_classes = len(train_generator.class_indices)
print(f"Number of classes detected: {num_classes}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(37, activation='softmax')  # 37 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
EPOCHS = 20

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

# Save the trained model
model.save(r"C:\Users\ASUS\Documents\sign_language_hindi\best_sign_model.h5")
print("Model trained and saved successfully!")

In [ ]:
import os

img_path = r"C:\Users\ASUS\Documents\sign_language_hindi\dataset\A\img1.jpg"
print("Exists?", os.path.exists(img_path))

In [ ]:
import os
print(os.path.exists(img_path))  # Should now print True

In [ ]:
class_names = sorted(SIGN_TO_HINDI.keys())

In [ ]:
import json
with open("class_names.json", "w") as f:
    json.dump(train_generator.class_indices, f)

In [ ]:
import json

with open("class_names.json", "r") as f:
    class_indices = json.load(f)

# Sort by index to get class_names in the order the model expects
class_names = [k for k, v in sorted(class_indices.items(), key=lambda x: x[1])]

In [ ]:
img = cv2.imread(r"dataset/A/A.a9d87ef1-2cd2-11f1-a225-505a65c8c938.jpg")
img_resized = cv2.resize(img, (64,64))
img_resized = img_resized / 255.0
img_resized = np.expand_dims(img_resized, axis=0)
pred = model.predict(img_resized)
pred_class = class_names[np.argmax(pred[0])]
pred_hindi = SIGN_TO_HINDI.get(pred_class, "Unknown")
print(pred_class, pred_hindi)

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ===============================
# Paths
# ===============================
DATASET_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\dataset"
MODEL_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\best_sign_model.h5"

# ===============================
# Load model
# ===============================
model = load_model(MODEL_PATH)
print("Model loaded!")

# ===============================
# Define class names in training order
# ===============================
class_names = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','BAD','HELLO','THANKYOU','YES','NO','PLEASE'
]

# Hindi mapping
SIGN_TO_HINDI = {
    'A': 'अ', 'B': 'ब', 'C': 'च', 'D': 'ड', 'E': 'ए',
    'F': 'फ', 'G': 'ग', 'H': 'ह', 'I': 'इ', 'J': 'ज',
    'K': 'क', 'L': 'ल', 'M': 'म', 'N': 'न', 'O': 'ओ',
    'P': 'प', 'Q': 'क्यू', 'R': 'र', 'S': 'स', 'T': 'ट',
    'U': 'उ', 'V': 'व', 'W': 'डब्ल्यू', 'X': 'एक्स', 'Y': 'य',
    'Z': 'ज़ेड', 'BAD':'बुरा', 'HELLO':'नमस्ते', 'THANKYOU':'धन्यवाद',
    'YES':'हाँ', 'NO':'नहीं', 'PLEASE':'कृपया'
}

# ===============================
# Loop through all classes
# ===============================
for class_folder_name in class_names:
    folder_path = os.path.join(DATASET_PATH, class_folder_name)
    
    if not os.path.exists(folder_path):
        print(f"Folder {class_folder_name} not found. Skipping...")
        continue
    
    # Loop through all images in this class
    for img_file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_file)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Cannot read image {img_file}. Skipping...")
            continue
        
        # Preprocess
        img_resized = cv2.resize(img, (64, 64))
        img_resized = img_resized / 255.0
        img_resized = np.expand_dims(img_resized, axis=0)
        
        # Predict
        pred = model.predict(img_resized)
        class_idx = np.argmax(pred[0])
        predicted_class = class_names[class_idx]
        predicted_hindi = SIGN_TO_HINDI.get(predicted_class, "Unknown")
        
        print(f"Image: {img_file} | Predicted class: {predicted_class} | Hindi: {predicted_hindi}")

In [2]:
import os
import numpy as np

DATASET_PATH = r"C:\Users\ASUS\Documents\sign_language_hindi\dataset"

# Get folder names (classes)
class_names = sorted(os.listdir(DATASET_PATH))

# Save file
np.save("class_names.npy", class_names)

print("Saved class names:", class_names)

Saved class names: ['.ipynb_checkpoints', 'A', 'B', 'C', 'D', 'E', 'F', 'FOOD', 'G', 'GOOD', 'H', 'HELLO', 'HELP', 'I', 'J', 'K', 'L', 'M', 'N', 'NO', 'O', 'P', 'PLEASE', 'Q', 'R', 'S', 'T', 'THANKYOU', 'U', 'V', 'W', 'WATER', 'X', 'Y', 'YES', 'Z']


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ===============================
# 1. Load Model + Classes
# ===============================
model = load_model("best_sign_model.h5")
class_names = np.load("class_names.npy", allow_pickle=True)

print("Model and classes loaded!")
font = ImageFont.truetype(r"C:\Users\ASUS\Documents\sign_language_hindi\Hindi.ttf", 32)
# ===============================
# 2. Sign → Hindi Mapping
# ===============================
SIGN_TO_HINDI = {
    'A': 'अ', 'B': 'ब', 'C': 'च', 'D': 'ड',
    'E': 'ए', 'F': 'फ', 'G': 'ग', 'H': 'ह',
    'I': 'इ', 'J': 'ज', 'K': 'क', 'L': 'ल',
    'M': 'म', 'N': 'न', 'O': 'ओ', 'P': 'प',
    'Q': 'क्यू', 'R': 'र', 'S': 'स', 'T': 'ट',
    'U': 'उ', 'V': 'व', 'W': 'डब्ल्यू', 'X': 'एक्स',
    'Y': 'य', 'Z': 'ज़',

    # Words (optional if trained)
    'HELLO': 'नमस्ते',
    'THANK YOU': 'धन्यवाद',
    'YES': 'हाँ',
    'NO': 'नहीं',
    'PLEASE': 'कृपया'
}

# ===============================
# 3. Prediction Function
# ===============================
def predict_sign(frame):
    img = cv2.resize(frame, (64, 64))
    img = img / 255.0
    img = np.reshape(img, (1, 64, 64, 3))

    pred = model.predict(img, verbose=0)
    index = np.argmax(pred)
    label = class_names[index]

    confidence = pred[0][index]

    return label, confidence

# ===============================
# 4. Start Webcam
# ===============================
cap = cv2.VideoCapture(0)

print("Press 'q' to exit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip for mirror view
    frame = cv2.flip(frame, 1)

    # ROI (center box)
    h, w, _ = frame.shape
    x1, y1 = int(w*0.3), int(h*0.2)
    x2, y2 = int(w*0.7), int(h*0.8)

    roi = frame[y1:y2, x1:x2]

    # Predict
    label, confidence = predict_sign(roi)

    # Hindi translation
    hindi = SIGN_TO_HINDI.get(label, "N/A")

    # ===============================
    # 5. Display
    # ===============================
    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    text = f"{label} ({confidence*100:.1f}%)"
    text_hindi = f"Hindi: {hindi}"

    cv2.putText(frame, text, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.putText(frame, text_hindi, (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

    cv2.imshow("Sign Language Translator", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Model and classes loaded!
Press 'q' to exit


In [11]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image, ImageDraw, ImageFont

# ===============================
# LOAD MODEL + CLASSES
# ===============================
model = load_model("best_sign_model.h5")
class_names = np.load("class_names.npy", allow_pickle=True)

# ===============================
# HINDI MAPPING
# ===============================
SIGN_TO_HINDI = {
    'A': 'अ', 'B': 'ब', 'C': 'च', 'D': 'ड',
    'E': 'ए', 'F': 'फ', 'G': 'ग', 'H': 'ह',
    'I': 'इ', 'J': 'ज', 'K': 'क', 'L': 'ल',
    'M': 'म', 'N': 'न', 'O': 'ओ', 'P': 'प',
    'Q': 'क्यू', 'R': 'र', 'S': 'स', 'T': 'ट',
    'U': 'उ', 'V': 'व', 'W': 'डब्ल्यू', 'X': 'एक्स',
    'Y': 'य', 'Z': 'ज़',
    'HELLO': 'नमस्ते',
    'THANK YOU': 'धन्यवाद'
}

# ===============================
# LOAD FONT
# ===============================
font = ImageFont.truetype("Hindi.ttf", 32)

# ===============================
# PREDICTION FUNCTION
# ===============================
def predict_sign(frame):
    img = cv2.resize(frame, (64, 64))
    img = img / 255.0
    img = np.reshape(img, (1, 64, 64, 3))

    pred = model.predict(img, verbose=0)
    index = np.argmax(pred)

    label = class_names[index]
    confidence = pred[0][index]

    return label, confidence

# ===============================
# START CAMERA
# ===============================
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # -------------------
    # CREATE ROI
    # -------------------
    h, w, _ = frame.shape
    x1, y1 = int(w * 0.3), int(h * 0.2)
    x2, y2 = int(w * 0.7), int(h * 0.8)

    roi = frame[y1:y2, x1:x2]

    # Draw box
    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    # -------------------
    # PREDICT
    # -------------------
    label, confidence = predict_sign(roi)

    label = str(label).upper().strip()
    hindi = SIGN_TO_HINDI.get(label, "N/A")

    print("Label:", label, "| Hindi:", hindi)

    # -------------------
    # ENGLISH TEXT
    # -------------------
    cv2.putText(frame, f"{label} ({confidence*100:.1f}%)",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    # -------------------
    # HINDI TEXT (PIL)
    # -------------------
    img_pil = Image.fromarray(frame)
    draw = ImageDraw.Draw(img_pil)

    draw.text((20, 80), f"Hindi: {hindi}", font=font, fill=(255, 0, 0))

    frame = np.array(img_pil)

    cv2.imshow("Sign Translator", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | Hindi: क्यू
Label: Q | 